In [25]:
import pandas as pd

#load the dataset
df = pd.read_csv(r"C:\Users\habee\Downloads\IMDB Dataset.csv\IMDB Dataset.csv")

In [26]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [27]:
#data exploration EDA
print("Shape:" , df.shape)
print("\nColumns:")
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nSentiment Distribution:")
print(df['sentiment'].value_counts())

Shape: (50000, 2)

Columns:
Index(['review', 'sentiment'], dtype='str')

Missing Values:
review       0
sentiment    0
dtype: int64

Sentiment Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


2. Text Preprocessing

In [28]:
import re 
import string
import nltk

from nltk.corpus import stopwords

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\habee\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [29]:
#Create a function to clean the text data
stop_words = set(stopwords.words('english'))
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r"<.*?>", "", text)  # Remove HTML tags
    text = text.translate(str.maketrans("", "", string.punctuation))  # Remove punctuation
    text = re.sub(r"\d+", "", text)  # Remove numbers
    text = re.sub(r"\s+", " ", text).strip()  # Remove extra whitespace 
    
    #remove stop words
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

In [30]:
sample_review = df['review'][0]

print("Original Review:\n")
print(sample_review)

print("\n" + "="*100 + "\n")

print("Cleaned Review:\n")
print(clean_text(sample_review))


Original Review:

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show

Apply it to the entire dataset

In [31]:
df["clean_review"] = df["review"].apply(clean_text)

In [32]:
df[["review", "clean_review", "sentiment"]].head()

,review,clean_review,sentiment
0,One of the other reviewers has mentioned that ...,one reviewers mentioned watching oz episode yo...,positive
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...,positive
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...,positive
3,Basically there's a family where a little boy ...,basically theres family little boy jake thinks...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...,positive


3. Feature Engineering (TF-IDF)

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [34]:
#create the vectorizer
vectorizer = TfidfVectorizer(max_features=5000)

In [35]:
X = vectorizer.fit_transform(df["clean_review"])

In [36]:
y = df["sentiment"]

In [37]:
print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Matrix Shape: (50000, 5000)
Target Shape: (50000,)


Now, split the data into training and testing

In [38]:
from sklearn.model_selection import train_test_split

In [39]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [40]:
print("Training data:",
      X_train.shape)
print("Testing data:",
      X_test.shape)

Training data: (40000, 5000)
Testing data: (10000, 5000)


Train the Logistic Regression model

In [41]:
from sklearn.linear_model import LogisticRegression

In [42]:
model = LogisticRegression()

In [43]:
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [44]:
print("Model trained successfully!")

Model trained successfully!


Model Evaluation

In [45]:
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score)
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8869



Classification Report:
               precision    recall  f1-score   support

    negative       0.90      0.87      0.88      4961
    positive       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


Confusion Matrix:
 [[4324  637]
 [ 494 4545]]


In [46]:
import joblib

joblib.dump(model, "../model/sentiment_model.pkl")
joblib.dump(vectorizer, "../model/tfidf_vectorizer.pkl")

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


In [47]:
sample = "amazing movie"

clean = clean_text(sample)
vec = vectorizer.transform([clean])

print("Cleaned:", clean)
print("Notebook prediction:", model.predict(vec)[0])

Cleaned: amazing movie
Notebook prediction: positive


In [48]:
model.classes_

array(['negative', 'positive'], dtype=object)

In [49]:
sample = "amazing movie"

clean = clean_text(sample)
print("Cleaned:", clean)

vec = vectorizer.transform([clean])

print("Prediction:", model.predict(vec)[0])
print("Probability:", model.predict_proba(vec))

Cleaned: amazing movie
Prediction: positive
Probability: [[0.01499064 0.98500936]]
